In [ ]:
import os
import gc
import math
import time
import torch
import pickle
import numpy as np

import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.autograd as autograd

import torch.nn.functional as functional
import torch.optim.lr_scheduler as lr_scheduler

from utils import *
os.makedirs('./xy_8x8', exist_ok=True)
torch.set_default_dtype(torch.float32) 
_ = torch.manual_seed(0)

In [ ]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [ ]:
if torch.cuda.is_available():
    torch_device = 'cuda'
else:
    torch_device = 'cpu'
print(f"TORCH DEVICE: {torch_device}")

In [ ]:
dim = 64
T = 1.40483871
data_filename = "8x8_lattices_at_t_1poin4_120000.pkl"
f = open(data_filename, 'rb')
dataset = pickle.load(f,encoding="latin1")
dataset = torch.tensor(dataset , dtype = torch.float32 , device = torch_device).reshape((-1,1,8,8))
data = dataset*(2*np.pi) # samples against temperature 1.40483871

In [ ]:
batch_size = 512
no_training_samples = 100000
no_validation_samples = 10000
no_test_samples = 10000

train_dataloader = torch.utils.data.DataLoader(data[:100000], batch_size = batch_size, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(data[100000:110000], batch_size = batch_size, shuffle=False)

test_dataloader = torch.utils.data.DataLoader(data[110000:120000], batch_size = batch_size, shuffle=False)

In [ ]:
model = RNVP(10,(1,8,8),256,1.,1.e-6).to(torch_device)

In [ ]:
PATH = "./xy_8x8/model_fkl_score_100000.pt"
optimizer = optim.Adam(model.parameters(),lr= 1.e-5)


In [ ]:
 # #For resuming the checkpoint:
 # checkpoint = torch.load(PATH)
 # model.load_state_dict(checkpoint['model_state_dict'])
 # optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [ ]:
import copy

patience = 25
patience_counter = 0
global_epoch = 0

# best_val_loss = 98.8109
# best_model_state = None

while patience_counter < patience:

    global_epoch += 1

    model.train()

    forward_loss = 0.0
    NLL = 0.0

    ########################################
    # Training
    ########################################
    for x in train_dataloader:

        x = x.to(torch_device)

        _, logq = model.inverse(x)

        fkl_loss = -torch.mean(logq)

        optimizer.zero_grad()
        fkl_loss.backward()
        optimizer.step()

        forward_loss += fkl_loss.item() * x.shape[0]

    ########################################
    # Validation
    ########################################
    model.eval()

    with torch.no_grad():

        for x in val_dataloader:

            x = x.to(torch_device)
            _, logq = model.inverse(x)
            nll = -torch.mean(logq)
            NLL += nll.item() * x.shape[0]

    mean_fkl = forward_loss / no_training_samples
    mean_nll = NLL / no_validation_samples

    ########################################
    # Early stopping
    ########################################
    if mean_nll < best_val_loss:

        best_val_loss = mean_nll
        patience_counter = 0

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

    else:
        patience_counter += 1

    ########################################
    # Logging
    ########################################
    print(
        f"Epoch {global_epoch:4d}, "
        f"FKL loss: {mean_fkl:.6f}, "
        f"Val loss: {mean_nll:.6f}, "
        f"Patience: {patience_counter}/{patience}"
    )

print(f"\nEarly stopping after {global_epoch} epochs.")
print(f"Best validation loss: {best_val_loss:.6f}")

model.load_state_dict(best_model_state)

In [ ]:
optimizer = optim.Adam(model.parameters(),lr= 1.e-5)

In [ ]:
no_test_samples  = 10000
NLL = 0.
model.eval()
for x in test_dataloader:
    z_, logq = model.inverse(x)
    nll = - torch.mean(logq)
    NLL += nll.item()*(x.shape[0])
test_nll = NLL/no_test_samples
print(f'Test NLL: {test_nll:.4f}')

In [ ]:
model.load_state_dict(best_model_state)

In [ ]:
torch.save({'model_state_dict': model.state_dict(),'optimizer_state_dict': optimizer.state_dict(),}, PATH)

In [ ]:
samples = []
neg_logq = []
train_dataloader = torch.utils.data.DataLoader(data[:100000], batch_size = 250, shuffle=True)

for x in train_dataloader:
    z1,logq = model.inverse(x)
    samples.append(x.cpu().detach().numpy())
    neg_logq.append(-logq.cpu().detach().numpy())


In [ ]:
samples = np.array(samples).reshape((-1,1,8,8))


In [ ]:
samples.shape

In [ ]:
neg_logq = np.array(neg_logq).reshape((-1))

In [ ]:
array = [samples,neg_logq]

In [ ]:
output = open("xy_8x8_nf_trained_data.pkl", 'wb')
pickle.dump(array, output)
output.close()